- Read bronze circuits table
- Keep only the columns required for analytics.
- standardise column names using snake_case.
- Concatenate name.givename and family name to create a new column called driver_name and transform the values to tile case.
- Filer out rows where circuit_id is null
- Remobe duplicate records.
- Transform values of columns circuit_name and locality to Title case
- write the trandformed data to silver circuits table

In [0]:
%run ../00.Common/01.environment-config


In [0]:
bronze_table = f"{catalog_nameone}.{bronze_schema}.results"
silver_table = f"{catalog_nameone}.{silver_schema}.results"

In [0]:
from pyspark.sql import functions as F

In [0]:
results_df = spark.read.table(bronze_table)

In [0]:
from pyspark.sql.functions import col

results_dropped_df = results_df.drop(F.col("url"))

In [0]:
results_renamed_df = (
  results_dropped_df
    .withColumnsRenamed({
      "constructorId": "constructor_Id",
      "driverId": "driver_Id",
      "raceName": "race_Name",
      "date": "race_date",
      "grid": "grid_position",
      "laps": "completed_laps",
      "number": "car_number",
      "position": "final_position",
      "positionText": "final_position_text"
    })
)

In [0]:
results_valid_df = (
    results_renamed_df
        .filter(
            F.col("season").isNotNull() &
            F.col("round").isNotNull() &
            F.col("driver_Id").isNotNull() &
            F.col("constructor_Id").isNotNull()
        )
)

In [0]:
display(results_renamed_df.count() - results_valid_df.count())

In [0]:
results_distinct_df = results_valid_df.dropDuplicates(["season", "round", "driver_Id", "constructor_Id"])

In [0]:
results_final_df = (
    results_distinct_df
        .withColumn('race_name', F.initcap(F.col("race_Name")))
)

In [0]:
results_final_df.write.format("delta").mode("overwrite").saveAsTable(silver_table)

In [0]:
display(spark.read.table(silver_table))